In [2]:
"""
Prep data for the few-shot BAT classification pipeline (Pipeline 1).

Inputs:
  merged_post40_llm.csv    (40 rows: 20 old + 20 new)
  merged_post40_Nadia.csv  (40 rows: 20 old + 20 new, same post order as LLM file)

Outputs:
  merged_post40_empty.csv  - merged_post40_llm.csv with EX/EMO/COG/MD/bat_score/
                              *_reasoning columns blanked out (a clean template)
  merged_post30_empty.csv  - merged_post40_empty.csv with the first 10 rows dropped
                              (these 30 are what the LLM will classify)
  few_shot_ex10.csv        - the first 10 rows of merged_post40_Nadia.csv
                              (Nadia's human labels used as the few-shot examples)

Row order in both merged_post40_* files is identical (same post_id sequence),
so "first 10" refers to the same 10 posts in both files.
"""

import pandas as pd

LLM_FILE = "merged_post40_llm.csv"
NADIA_FILE = "merged_post40_Nadia.csv"

EMPTY_COLS = [
    "EX", "EMO", "COG", "MD", "bat_score",
    "EX_reasoning", "EMO_reasoning", "COG_reasoning", "MD_reasoning",
]

llm_df = pd.read_csv(LLM_FILE)
nadia_df = pd.read_csv(NADIA_FILE)

# Sanity check: same post order in both files before we rely on "first 10" meaning
# the same 10 posts in each.
llm_ids = llm_df["post_id"].astype(str).tolist()
nadia_ids = nadia_df["post_id"].astype(str).tolist()
if llm_ids[:20] != nadia_ids[:20]:
    print("WARNING: first 20 post_ids differ between LLM and Nadia files!")
    print("  LLM  :", llm_ids[:20])
    print("  Nadia:", nadia_ids[:20])
else:
    print("OK: first 20 post_ids match between LLM and Nadia files.")

# ── 1. merged_post40_empty.csv ────────────────────────────────────────────────
empty_df = llm_df.copy()
for col in EMPTY_COLS:
    if col in empty_df.columns:
        empty_df[col] = ""
    else:
        empty_df[col] = ""  # add if missing, e.g. Nadia-style files lack some
empty_df.to_csv("merged_post20_empty.csv", index=False)
print(f"Saved merged_post20_empty.csv ({len(empty_df)} rows)")

# ── 2. merged_post30_empty.csv (drop first 10 rows) ──────────────────────────
post30_df = empty_df.iloc[20:].reset_index(drop=True)
post30_df.to_csv("merged_post20_empty.csv", index=False)
print(f"Saved merged_post20_empty.csv ({len(post30_df)} rows)")

# ── 3. few_shot_ex10.csv (first 10 rows of Nadia's labels) ───────────────────
fewshot_df = nadia_df.iloc[:20].reset_index(drop=True)
fewshot_df.to_csv("few_shot_ex20.csv", index=False)
print(f"Saved few_shot_ex20.csv ({len(fewshot_df)} rows)")

print("\nDone.")

OK: first 20 post_ids match between LLM and Nadia files.
Saved merged_post20_empty.csv (40 rows)
Saved merged_post20_empty.csv (20 rows)
Saved few_shot_ex20.csv (20 rows)

Done.
